# Replicating two served numbers from raw data

This notebook asks the platform for two numbers through the Python client, then rebuilds each
from the raw generated data with NumPy and asserts they are equal:

1. the firm's 99% one-day VaR (`var_99_1d`), from the raw trade P&L vectors (`pnl_vectors`);
2. one FRTB standardised-approach equity delta bucket charge `K_b`, from the raw
   sensitivities, against the curated `frtb_sa_detail` row.

It generates and runs the `tiny` profile into a temporary directory first (about half a
minute; Spark runs in local mode), so it needs no Docker and no running server. The client
runs in-process: that run is never loaded into Pinot, so the client answers from the curated
Parquet files and says so in `client.source`. The temporary directory is deleted at the end.

In [ ]:
import shutil
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

from riskagg import conventions
from riskagg.cli import generate, run
from riskagg.paths import frtb_sa_path
from riskagg.semantic.model import load_model
from riskagg.serving.client import RiskClient

AS_OF = "2026-11-12"
ROOT = Path(tempfile.mkdtemp(prefix="riskagg-notebook-"))
pd.options.display.float_format = "{:,.2f}".format

## 1. Generate and run the tiny profile

`generate` writes the raw datasets; `run` aggregates them, writes the curated tables, runs
the reconciliation checks and marks the run official. These are the same functions as
`riskagg generate` and `riskagg run` on the command line.

In [ ]:
generate(profile="tiny", as_of=AS_OF, root=ROOT)
run(profile="tiny", as_of=AS_OF, root=ROOT)

## 2. Firm VaR through the client

`RiskClient()` with no URL runs the service in-process. The answer comes from the official
run of the as-of date; `source`, `as_of_date` and `run_id` say what answered.

In [ ]:
client = RiskClient(root=ROOT)
firm = client.metric("var_99_1d", "firm_id", as_of=AS_OF)
print(f"source={client.source} as_of={client.as_of_date} run_id={client.run_id}")
firm

## 3. The same VaR from the raw trade vectors

Sum every trade's CURRENT-window P&L vector scenario by scenario, then apply the quantile
convention from `riskagg.conventions` (the only place it lives): the VaR is the
`max(1, ceil(n x (1 - level)))`-th largest loss. VaRs are never added.

In [ ]:
model = load_model()
measure = model.measure("var_99_1d")
raw = ROOT / "raw" / "tiny" / f"as_of_date={AS_OF}"

vectors = pd.read_parquet(raw / "pnl_vectors")
current = vectors[vectors["window"] == measure.params["window"]]
firm_pnl = np.sum(np.stack(current["pnl"].to_numpy()), axis=0)
level = measure.params["level"]
rebuilt = conventions.var_from_pnl(firm_pnl, level)
served = float(firm["value"].iloc[0])

n = len(firm_pnl)
print(f"{len(current)} trade vectors of {n} scenarios; VaR is loss rank {conventions.var_rank(n, level)}")
print(f"served {served:,.2f}  rebuilt {rebuilt:,.2f}")
assert np.isclose(served, rebuilt, rtol=1e-9, atol=1e-6)
print("firm var_99_1d reproduced")

Adding up the desks' VaRs gives a different (here larger) number: each desk's VaR comes
from its own worst scenarios, and desks offset each other within a scenario.

In [ ]:
desks = client.metric("var_99_1d", "desk_id", as_of=AS_OF)
print(desks.to_string(index=False))
print(f"sum of desk VaRs {desks['value'].sum():,.2f} against firm VaR {served:,.2f}")

## 4. One FRTB SA equity delta bucket from raw sensitivities

For one equity bucket at the firm node (MAR21.4):

- net the delta sensitivities per risk factor across all trades;
- weight each by the bucket's risk weight: `WS_k = s_k x RW_b`;
- `K_b = sqrt(max(0, sum_k sum_l rho_kl WS_k WS_l))` with `rho_kk = 1` and, between two
  factors of the bucket, the bucket's `rho` under each correlation scenario (MAR21.6:
  high `min(1.25 rho, 1)`, low `max(2 rho - 1, 0.75 rho)`);
- `S_b = sum_k WS_k`.

The risk weights and correlations are read from `config/reference/frtb_sa.yaml`. The bucket
picked is the one with the most delta sensitivities. The curated value is read through the
client's drill-down on `frtb_sa_detail`.

On the tiny profile each equity bucket holds one issuer, so the netting across trades and
desks is what this checks: with one factor `K_b = |S_b|` and `rho` does not enter. The code
below is the general formula and applies unchanged to a bucket with several factors.

In [ ]:
ref = yaml.safe_load(frtb_sa_path().read_text())
eq = ref["EQ"]["delta"]["buckets"]
high, low = ref["correlation_scenarios"]["high"], ref["correlation_scenarios"]["low"]
scenario_rho = {
    "LOW": lambda r: max(low["slope"] * r + low["intercept"], low["floor_multiplier"] * r),
    "MEDIUM": lambda r: r,
    "HIGH": lambda r: min(high["multiplier"] * r, high["cap"]),
}

sens = pd.read_parquet(raw / "sensitivities")
delta = sens[(sens["risk_class"] == "EQ") & (sens["factor_type"] == "DELTA")]
bucket = delta["bucket"].value_counts().sort_index().idxmax()
net = delta[delta["bucket"] == bucket].groupby("risk_factor_id")["delta"].sum()
ws = net.to_numpy() * eq[bucket]["risk_weight"]
print(f"bucket {bucket}: {len(delta[delta['bucket'] == bucket])} sensitivities, "
      f"{len(ws)} risk factor(s), risk weight {eq[bucket]['risk_weight']}, rho {eq[bucket]['rho']}")

curated = client.drilldown(
    "frtb_sa_detail", as_of=AS_OF, node_level="firm_id", risk_class="EQ",
    component="DELTA", bucket=bucket,
).set_index("corr_scenario")
print(f"source={client.source}")

for scenario, adjust in scenario_rho.items():
    rho = np.full((len(ws), len(ws)), adjust(eq[bucket]["rho"]))
    np.fill_diagonal(rho, 1.0)
    k_b = float(np.sqrt(max(0.0, ws @ rho @ ws)))
    s_b = float(ws.sum())
    row = curated.loc[scenario]
    print(f"{scenario:<6} K_b served {row['k_b']:,.2f} rebuilt {k_b:,.2f}   "
          f"S_b served {row['s_b']:,.2f} rebuilt {s_b:,.2f}")
    assert np.isclose(row["k_b"], k_b, rtol=1e-9) and np.isclose(row["s_b"], s_b, rtol=1e-9)
print(f"equity delta bucket {bucket} K_b reproduced under every correlation scenario")

## 5. Clean up

In [ ]:
shutil.rmtree(ROOT)